# Q4 Ceiling-Aware UdonPred Ensembles

Question: can ensembles of UdonPred's seven dataset-specific heads outperform individual heads, and does annotation-ceiling information help choose which heads to trust for each target dataset?

This notebook extends the local UdonPred ensemble analysis with two Q3/Q4 connections. First, it adds a ceiling-aware validation-weighted ensemble that combines validation headroom above simple baselines with inter-dataset annotation compatibility. Second, it re-expresses ensemble performance as normalized headroom above the best simple baseline. External CAID methods remain context only; this notebook uses local UdonPred predictions.

Learned weights and stacking models are fit on validation predictions, then evaluated on held-out test predictions. The ceiling-aware strategy should be treated as exploratory when the ceiling estimates come from test-set annotation overlaps.


## 1. Setup

In [14]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from torchmetrics.functional import auroc, average_precision, spearman_corrcoef

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

UDONPRED_DIR = ROOT / "UdonPred"
RESULTS = ROOT / "results"
TEST_PREDICTION_ROOT = RESULTS / "udonpred_matrix" / "predictions"
VALID_PREDICTION_ROOT = RESULTS / "udonpred_validation_matrix" / "predictions"
ENSEMBLE_DIR = RESULTS / "ensembles"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]
NEGATED_DATASETS = {"chezod", "plddt"}
MASK_VALUE = 999.0
METRIC_COLS = [
    "trizod",
    "chezod",
    "softdis",
    "pdbflex",
    "atlas",
    "plddt",
    "disprot\n(AP)",
    "disprot\n(AUROC)",
]

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")


## 2. Shared Helpers

The helpers below mirror the earlier notebooks: CheZOD and pLDDT are oriented so larger values mean more disorder, masked labels are excluded, continuous targets use Spearman correlation, and DisProt uses AP/AUROC.

In [ ]:
def read_jsonl_records(path: Path) -> dict[str, dict[str, object]]:
    records = {}
    with path.open() as handle:
        for line in handle:
            raw = json.loads(line)
            records[str(raw["id"])] = {
                "sequence": str(raw["x_0"]),
                "labels": np.asarray(raw["y"], dtype=np.float64),
            }
    return records


def normalize_prediction_id(protein_id: str) -> str:
    protein_id = protein_id.strip().lstrip(">")
    if len(protein_id) >= 6 and protein_id.isdigit():
        base_len = len(protein_id) - 3
        return protein_id[:base_len] + "_" + "_".join(protein_id[base_len:])
    return protein_id


def read_caid_dir(input_dir: Path) -> dict[str, np.ndarray]:
    predictions = {}
    for path in sorted(input_dir.glob("*.caid")):
        with path.open() as handle:
            lines = handle.readlines()
        if not lines:
            raise ValueError(f"Empty prediction file: {path}")
        protein_id = normalize_prediction_id(lines[0])
        scores = []
        for line in lines[1:]:
            line = line.strip()
            if line:
                scores.append(float(line.split("\t")[2]))
        predictions[protein_id] = np.asarray(scores, dtype=np.float64)
    return predictions


def count_fasta_records(path: Path) -> int:
    return sum(1 for line in path.read_text().splitlines() if line.startswith(">"))


def prediction_dir_complete(result_dir: Path, fasta_path: Path) -> bool:
    return result_dir.exists() and len(list(result_dir.glob("*.caid"))) == count_fasta_records(fasta_path)


def labels_in_disorder_direction(labels: np.ndarray, dataset: str) -> np.ndarray:
    return -labels if dataset in NEGATED_DATASETS else labels


def preds_in_disorder_direction(preds: np.ndarray, train_dataset: str) -> np.ndarray:
    return -preds if train_dataset in NEGATED_DATASETS else preds


def metric_columns_for_dataset(dataset: str) -> list[str]:
    if dataset == "disprot":
        return ["disprot\n(AP)", "disprot\n(AUROC)"]
    return [dataset]


def evaluate_vector(labels: np.ndarray, preds: np.ndarray, dataset: str) -> dict[str, float]:
    if len(labels) == 0:
        return {column: math.nan for column in metric_columns_for_dataset(dataset)}
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    preds_tensor = torch.tensor(preds, dtype=torch.float32)
    if dataset == "disprot":
        binary_labels = labels_tensor.to(torch.int)
        return {
            "disprot\n(AP)": float(average_precision(preds_tensor, binary_labels, task="binary")),
            "disprot\n(AUROC)": float(auroc(preds_tensor, binary_labels, task="binary")),
        }
    if np.unique(labels).size < 2 or np.unique(preds).size < 2:
        return {dataset: math.nan}
    return {dataset: float(spearman_corrcoef(preds_tensor, labels_tensor))}


def primary_metric(dataset: str) -> str:
    return "disprot\n(AP)" if dataset == "disprot" else dataset


def load_aligned_stack(split: str, prediction_root: Path, target_dataset: str) -> dict[str, object]:
    records = read_jsonl_records(UDONPRED_DIR / "data" / target_dataset / f"{split}.jsonl")
    pred_maps = {
        train_dataset: read_caid_dir(prediction_root / f"{train_dataset}_{target_dataset}")
        for train_dataset in DATASETS
    }

    label_chunks = []
    pred_chunks_by_train = {train_dataset: [] for train_dataset in DATASETS}
    residue_count = 0
    for protein_id, record in records.items():
        labels = np.asarray(record["labels"], dtype=np.float64)
        mask = np.isfinite(labels) & (labels != MASK_VALUE)
        if not np.any(mask):
            continue

        for train_dataset, pred_map in pred_maps.items():
            if protein_id not in pred_map:
                raise ValueError(f"Missing prediction for {protein_id} in {train_dataset}_{target_dataset}")
            preds = pred_map[protein_id]
            if len(preds) != len(labels):
                raise ValueError(
                    f"{train_dataset}_{target_dataset}/{protein_id}: prediction length {len(preds)} "
                    f"!= label length {len(labels)}"
                )
            pred_chunks_by_train[train_dataset].append(preds_in_disorder_direction(preds, train_dataset)[mask])

        label_chunks.append(labels_in_disorder_direction(labels, target_dataset)[mask])
        residue_count += int(mask.sum())

    y = np.concatenate(label_chunks)
    x = np.column_stack([np.concatenate(pred_chunks_by_train[train_dataset]) for train_dataset in DATASETS])
    return {"dataset": target_dataset, "split": split, "X": x, "y": y, "n_residues": residue_count}


def load_split_stacks(split: str, prediction_root: Path) -> dict[str, dict[str, object]]:
    return {
        target_dataset: load_aligned_stack(split, prediction_root, target_dataset)
        for target_dataset in DATASETS
    }


## 3. Generate Missing Validation Predictions

Learned ensembles must be fit on validation labels, not test labels. This cell creates the validation prediction matrix if it is missing. It skips complete prediction directories.

In [ ]:
RUN_VALIDATION_PREDICTIONS = True
MPS_HIGH_WATERMARK_RATIO = "0.0"
MPS_LOW_WATERMARK_RATIO = "0.0"
if torch.cuda.is_available():
    PREDICTION_DEVICE = "cuda"
elif torch.backends.mps.is_available():
    PREDICTION_DEVICE = "mps"
else:
    PREDICTION_DEVICE = "cpu"

if PREDICTION_DEVICE == "cuda":
    PREDICTION_BATCH_SIZE = "2000"
elif PREDICTION_DEVICE == "mps":
    PREDICTION_BATCH_SIZE = "2000"
else:
    PREDICTION_BATCH_SIZE = "200"
PREDICTION_PYTHON = UDONPRED_DIR / ".venv" / "bin" / "python"
if not PREDICTION_PYTHON.exists():
    PREDICTION_PYTHON = Path(sys.executable)

print(f"Validation prediction device: {PREDICTION_DEVICE}")
print(f"Validation prediction batch size: {PREDICTION_BATCH_SIZE}")
print(f"Validation prediction Python: {PREDICTION_PYTHON}")
if PREDICTION_DEVICE == "mps":
    print(f"PYTORCH_MPS_HIGH_WATERMARK_RATIO={MPS_HIGH_WATERMARK_RATIO}")
    print(f"PYTORCH_MPS_LOW_WATERMARK_RATIO={MPS_LOW_WATERMARK_RATIO}")

missing_validation_jobs = []
for train_dataset in DATASETS:
    for target_dataset in DATASETS:
        fasta_path = UDONPRED_DIR / "data" / target_dataset / "valid.fasta"
        result_dir = VALID_PREDICTION_ROOT / f"{train_dataset}_{target_dataset}"
        if not prediction_dir_complete(result_dir, fasta_path):
            missing_validation_jobs.append((train_dataset, target_dataset, fasta_path, result_dir))

print(f"Missing validation prediction jobs: {len(missing_validation_jobs)}")

if RUN_VALIDATION_PREDICTIONS and missing_validation_jobs:
    cmd = [
        str(PREDICTION_PYTHON),
        str(ROOT / "scripts" / "run_udonpred_matrix_fast.py"),
        "--udonpred-dir",
        str(UDONPRED_DIR),
        "--output-dir",
        str(RESULTS / "udonpred_validation_matrix"),
        "--split",
        "valid",
        "--device",
        PREDICTION_DEVICE,
        "--batch-size",
        PREDICTION_BATCH_SIZE,
        "--smooth",
        "1.5",
        "--predictions-only",
    ]
    env = None
    if PREDICTION_DEVICE == "mps":
        env = {
            **os.environ,
            "PYTORCH_MPS_HIGH_WATERMARK_RATIO": MPS_HIGH_WATERMARK_RATIO,
            "PYTORCH_MPS_LOW_WATERMARK_RATIO": MPS_LOW_WATERMARK_RATIO,
        }
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True, env=env)
elif missing_validation_jobs:
    print("Set RUN_VALIDATION_PREDICTIONS = True to generate missing validation predictions.")
else:
    print("Validation prediction matrix is complete.")


## 4. Load And Sanity-Check Aligned Predictions

In [ ]:
test_stacks = load_split_stacks("test", TEST_PREDICTION_ROOT)
valid_stacks = load_split_stacks("valid", VALID_PREDICTION_ROOT)

alignment_summary = pd.DataFrame(
    [
        {"split": "test", "dataset": dataset, "n_residues": stack["n_residues"], "n_heads": stack["X"].shape[1]}
        for dataset, stack in test_stacks.items()
    ]
    + [
        {"split": "valid", "dataset": dataset, "n_residues": stack["n_residues"], "n_heads": stack["X"].shape[1]}
        for dataset, stack in valid_stacks.items()
    ]
)
display(alignment_summary)


In [ ]:
def score_individual_heads(stacks: dict[str, dict[str, object]]) -> pd.DataFrame:
    rows = []
    for train_index, train_dataset in enumerate(DATASETS):
        row = {"train_dataset": train_dataset}
        for target_dataset, stack in stacks.items():
            row.update(evaluate_vector(stack["y"], stack["X"][:, train_index], target_dataset))
        rows.append(row)
    return pd.DataFrame(rows).set_index("train_dataset")[METRIC_COLS]


recomputed_individual = score_individual_heads(test_stacks)
published_individual = pd.read_csv(RESULTS / "udonpred_matrix" / "matrix.csv").set_index("train_dataset")[METRIC_COLS]
published_individual = published_individual.apply(pd.to_numeric)

max_abs_delta = (recomputed_individual - published_individual).abs().max().max()
print(f"Max absolute delta versus results/udonpred_matrix/matrix.csv: {max_abs_delta:.6f}")
if max_abs_delta > 5e-4:
    raise ValueError("Recomputed individual-head matrix does not match the saved UdonPred matrix.")

best_individual_scores = recomputed_individual.max(axis=0)
best_individual_heads = recomputed_individual.idxmax(axis=0)
display(recomputed_individual.style.format("{:.3f}"))

Max absolute delta versus results/udonpred_matrix/matrix.csv: 0.000016


,trizod,chezod,softdis,pdbflex,atlas,plddt,disprot (AP),disprot (AUROC)
train_dataset,,,,,,,,
trizod,0.508,0.708,0.257,-0.006,0.413,0.719,0.899,0.930
chezod,0.486,0.691,0.235,-0.039,0.327,0.713,0.890,0.925
softdis,0.470,0.537,0.500,0.150,0.597,0.784,0.872,0.918
pdbflex,0.287,0.286,0.292,0.395,0.349,0.520,0.519,0.737
atlas,0.400,0.595,0.310,0.056,0.708,0.719,0.870,0.920
plddt,0.423,0.625,0.284,0.104,0.512,0.839,0.875,0.927
disprot,0.364,0.663,0.255,0.099,0.624,0.774,0.935,0.959


## 5. Fit Non-Leaky Ensembles On Validation Data

In [ ]:
def concatenate_stacks(stacks: list[dict[str, object]]) -> tuple[np.ndarray, np.ndarray]:
    x = np.vstack([stack["X"] for stack in stacks])
    y = np.concatenate([stack["y"] for stack in stacks])
    return x, y


def fit_convex_weights(stacks: list[dict[str, object]]) -> np.ndarray:
    x, y = concatenate_stacks(stacks)
    n_heads = x.shape[1]
    start = np.full(n_heads, 1.0 / n_heads)
    constraints = ({"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)},)
    bounds = [(0.0, 1.0)] * n_heads

    def objective(weights: np.ndarray) -> float:
        residual = x @ weights - y
        return float(np.mean(residual * residual))

    result = minimize(objective, start, method="SLSQP", bounds=bounds, constraints=constraints)
    if not result.success:
        raise RuntimeError(f"Convex-weight optimization failed: {result.message}")
    weights = np.asarray(result.x, dtype=np.float64)
    weights[weights < 1e-10] = 0.0
    return weights / weights.sum()


def fit_ridge_model(stacks: list[dict[str, object]], alpha: float = 1.0) -> Ridge:
    x, y = concatenate_stacks(stacks)
    model = Ridge(alpha=alpha)
    model.fit(x, y)
    return model


def normalize_positive_weights(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    values[values < 0.0] = 0.0
    total = values.sum()
    if total <= 0.0:
        return np.full(values.shape, 1.0 / len(values), dtype=np.float64)
    return values / total


def load_best_simple_baseline_scores() -> pd.Series:
    path = RESULTS / "normalized_headroom" / "best_simple_baseline_per_metric.csv"
    if path.exists():
        summary = pd.read_csv(path)
        return summary.set_index("test_metric")["best_simple_baseline_score"].reindex(METRIC_COLS).apply(pd.to_numeric, errors="coerce")

    baseline_matrix = pd.read_csv(RESULTS / "simple_baselines" / "matrix.csv")
    baseline_scores = baseline_matrix.set_index(["baseline", "train_dataset"])[METRIC_COLS]
    return baseline_scores.apply(pd.to_numeric, errors="coerce").max(axis=0)


def load_ceiling_matrix() -> pd.DataFrame:
    path = RESULTS / "normalized_headroom" / "ceiling_matrix.csv"
    if not path.exists():
        path = RESULTS / "annotation_ceiling" / "ceiling_matrix.csv"
    if not path.exists():
        raise FileNotFoundError(
            "Missing ceiling matrix. Run scripts/compute_normalized_headroom.py before this notebook."
        )
    return pd.read_csv(path, index_col=0).reindex(index=DATASETS, columns=METRIC_COLS).apply(pd.to_numeric, errors="coerce")


def ceiling_aware_weights_for_target(target_dataset: str) -> np.ndarray:
    metric = primary_metric(target_dataset)
    validation_scores = valid_individual[metric].reindex(DATASETS).to_numpy(dtype=np.float64)
    baseline_score = float(best_simple_baseline_scores[metric])
    validation_signal = validation_scores - baseline_score
    validation_signal = np.nan_to_num(validation_signal, nan=0.0, posinf=0.0, neginf=0.0)

    ceiling_signal = ceiling_matrix[metric].reindex(DATASETS).to_numpy(dtype=np.float64)
    ceiling_signal = np.nan_to_num(ceiling_signal, nan=0.5, posinf=0.5, neginf=0.0)
    ceiling_signal[ceiling_signal < 0.0] = 0.0
    ceiling_signal[DATASETS.index(target_dataset)] = 1.0

    return normalize_positive_weights(validation_signal * ceiling_signal)


valid_individual = score_individual_heads(valid_stacks)
selected_head_by_target = {
    dataset: valid_individual[primary_metric(dataset)].idxmax()
    for dataset in DATASETS
}

ceiling_matrix = load_ceiling_matrix()
best_simple_baseline_scores = load_best_simple_baseline_scores()

global_convex_weights = fit_convex_weights([valid_stacks[dataset] for dataset in DATASETS])
per_target_convex_weights = {
    dataset: fit_convex_weights([valid_stacks[dataset]])
    for dataset in DATASETS
}
per_target_ceiling_aware_weights = {
    dataset: ceiling_aware_weights_for_target(dataset)
    for dataset in DATASETS
}

global_ridge = fit_ridge_model([valid_stacks[dataset] for dataset in DATASETS])
per_target_ridge = {
    dataset: fit_ridge_model([valid_stacks[dataset]])
    for dataset in DATASETS
}

weight_rows = [
    {"strategy": "global_convex_validation", "target_dataset": "all", **dict(zip(DATASETS, global_convex_weights))}
]
weight_rows += [
    {"strategy": "per_target_convex_validation", "target_dataset": dataset, **dict(zip(DATASETS, weights))}
    for dataset, weights in per_target_convex_weights.items()
]
weight_rows += [
    {"strategy": "per_target_ceiling_aware_validation", "target_dataset": dataset, **dict(zip(DATASETS, weights))}
    for dataset, weights in per_target_ceiling_aware_weights.items()
]
weights_df = pd.DataFrame(weight_rows)
weights_df.to_csv(ENSEMBLE_DIR / "ensemble_weights.csv", index=False)

print("Validation-selected head by target:")
display(pd.Series(selected_head_by_target, name="selected_train_dataset").to_frame())
display(weights_df.style.format({dataset: "{:.3f}" for dataset in DATASETS}))


## 6. Evaluate Ensemble Strategies On Test Sets

In [ ]:
def strategy_predictions_for_dataset(strategy: str, dataset: str, stack: dict[str, object]) -> np.ndarray:
    x = stack["X"]
    if strategy == "simple_mean_all_heads":
        return x.mean(axis=1)
    if strategy == "validation_selected_single_head":
        return x[:, DATASETS.index(selected_head_by_target[dataset])]
    if strategy == "global_convex_validation":
        return x @ global_convex_weights
    if strategy == "per_target_convex_validation":
        return x @ per_target_convex_weights[dataset]
    if strategy == "per_target_ceiling_aware_validation":
        return x @ per_target_ceiling_aware_weights[dataset]
    if strategy == "global_ridge_stacking_validation":
        return global_ridge.predict(x)
    if strategy == "per_target_ridge_stacking_validation":
        return per_target_ridge[dataset].predict(x)
    raise ValueError(f"Unknown strategy: {strategy}")


strategies = [
    "best_individual_test_oracle",
    "simple_mean_all_heads",
    "validation_selected_single_head",
    "global_convex_validation",
    "per_target_convex_validation",
    "per_target_ceiling_aware_validation",
    "global_ridge_stacking_validation",
    "per_target_ridge_stacking_validation",
]

rows = []
for strategy in strategies:
    row = {"strategy": strategy}
    if strategy == "best_individual_test_oracle":
        row.update(best_individual_scores.to_dict())
    else:
        for dataset, stack in test_stacks.items():
            preds = strategy_predictions_for_dataset(strategy, dataset, stack)
            row.update(evaluate_vector(stack["y"], preds, dataset))
    rows.append(row)

ensemble_matrix = pd.DataFrame(rows).set_index("strategy")[METRIC_COLS]
ensemble_delta = ensemble_matrix.subtract(best_individual_scores, axis="columns")

summary_rows = []
for strategy in ensemble_matrix.index:
    deltas = ensemble_delta.loc[strategy]
    summary_rows.append(
        {
            "strategy": strategy,
            "mean_score": float(ensemble_matrix.loc[strategy].mean()),
            "mean_delta_vs_best_individual": float(deltas.mean()),
            "wins_vs_best_individual": int((deltas > 1e-6).sum()),
            "ties_vs_best_individual": int((deltas.abs() <= 1e-6).sum()),
            "losses_vs_best_individual": int((deltas < -1e-6).sum()),
            "best_delta": float(deltas.max()),
            "worst_delta": float(deltas.min()),
        }
    )
ensemble_summary = pd.DataFrame(summary_rows).set_index("strategy")

ensemble_matrix.to_csv(ENSEMBLE_DIR / "ensemble_matrix.csv")
ensemble_delta.to_csv(ENSEMBLE_DIR / "ensemble_delta_vs_best_individual.csv")
ensemble_summary.to_csv(ENSEMBLE_DIR / "ensemble_summary.csv")
valid_individual.to_csv(ENSEMBLE_DIR / "validation_individual_matrix.csv")

display(ensemble_matrix.style.format("{:.3f}"))
display(ensemble_delta.style.format("{:+.3f}"))
display(ensemble_summary.style.format("{:.3f}"))


## 7. Normalized Headroom

Re-express ensemble scores as headroom above the best simple baseline. Since an ensemble is evaluated against each target dataset's own labels, the target-side ceiling is 1.0 for each metric; off-diagonal annotation ceilings are used separately in the ceiling-aware weighting strategy above.


In [ ]:
ensemble_available_headroom = 1.0 - best_simple_baseline_scores
ensemble_raw_headroom = ensemble_matrix.subtract(best_simple_baseline_scores, axis="columns")
ensemble_normalized_headroom = ensemble_raw_headroom.divide(ensemble_available_headroom, axis="columns")
invalid_headroom_cols = ensemble_available_headroom <= 1e-12
ensemble_normalized_headroom.loc[:, invalid_headroom_cols] = np.nan

headroom_summary_rows = []
for strategy in ensemble_normalized_headroom.index:
    values = ensemble_normalized_headroom.loc[strategy]
    headroom_summary_rows.append(
        {
            "strategy": strategy,
            "mean_normalized_headroom": float(values.mean()),
            "metrics_above_baseline": int((ensemble_raw_headroom.loc[strategy] > 1e-6).sum()),
            "metrics_above_best_individual_headroom": int(
                ((ensemble_normalized_headroom.loc[strategy] - ensemble_normalized_headroom.loc["best_individual_test_oracle"]) > 1e-6).sum()
            ),
            "best_normalized_headroom": float(values.max()),
            "worst_normalized_headroom": float(values.min()),
        }
    )
ensemble_headroom_summary = pd.DataFrame(headroom_summary_rows).set_index("strategy")

ensemble_raw_headroom.to_csv(ENSEMBLE_DIR / "ensemble_raw_headroom_vs_best_simple_baseline.csv")
ensemble_normalized_headroom.to_csv(ENSEMBLE_DIR / "ensemble_normalized_headroom_vs_best_simple_baseline.csv")
ensemble_headroom_summary.to_csv(ENSEMBLE_DIR / "ensemble_headroom_summary.csv")

print("Best simple baseline scores used for ensemble headroom:")
display(best_simple_baseline_scores.to_frame("best_simple_baseline_score").T.style.format("{:.3f}"))
display(ensemble_normalized_headroom.style.format("{:.3f}"))
display(ensemble_headroom_summary.style.format("{:.3f}"))


## 8. Plots


In [ ]:
def save_heatmap(matrix: pd.DataFrame, path: Path, title: str, cmap: str = "viridis", center: float | None = None, fmt: str = ".3f") -> None:
    if matrix.empty or not matrix.notna().any().any():
        print(f"Skipping {path.name}: no finite values to plot.")
        return
    plt.figure(figsize=(13, max(4, 0.45 * len(matrix))))
    sns.heatmap(matrix, annot=True, fmt=fmt, cmap=cmap, center=center, linewidths=0.5, linecolor="white")
    plt.xlabel("Test dataset / metric")
    plt.ylabel("Strategy")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()


save_heatmap(
    ensemble_matrix,
    ENSEMBLE_DIR / "ensemble_matrix_heatmap.png",
    "Q4 ensemble performance",
)

save_heatmap(
    ensemble_normalized_headroom,
    ENSEMBLE_DIR / "ensemble_normalized_headroom_heatmap.png",
    "Normalized ensemble headroom above best simple baseline",
)

plot_delta = ensemble_delta.drop(index="best_individual_test_oracle")
save_heatmap(
    plot_delta,
    ENSEMBLE_DIR / "ensemble_delta_vs_best_individual_heatmap.png",
    "Delta versus best individual UdonPred head",
    cmap="coolwarm",
    center=0,
    fmt="+.3f",
)

summary_plot = ensemble_summary.drop(index="best_individual_test_oracle").reset_index()
plt.figure(figsize=(10, 4))
sns.barplot(data=summary_plot, x="strategy", y="mean_delta_vs_best_individual", color="#4C78A8")
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=35, ha="right")
plt.xlabel("Strategy")
plt.ylabel("Mean delta")
plt.title("Mean ensemble delta versus best individual head")
plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / "ensemble_mean_delta_barplot.png", dpi=200)
plt.show()

# Ceiling-focused plots: these are the figures to use when discussing the ceiling integration.
ceiling_focus_strategies = [
    "best_individual_test_oracle",
    "validation_selected_single_head",
    "per_target_convex_validation",
    "per_target_ceiling_aware_validation",
    "per_target_ridge_stacking_validation",
]
ceiling_focus_strategies = [strategy for strategy in ceiling_focus_strategies if strategy in ensemble_matrix.index]

ceiling_focus_labels = {
    "best_individual_test_oracle": "Best individual\n(test oracle)",
    "validation_selected_single_head": "Validation-selected\nsingle head",
    "per_target_convex_validation": "Per-target\nconvex",
    "per_target_ceiling_aware_validation": "Ceiling-aware\nvalidation",
    "per_target_ridge_stacking_validation": "Per-target\nridge",
}

focused_headroom = ensemble_normalized_headroom.loc[ceiling_focus_strategies].rename(index=ceiling_focus_labels)
save_heatmap(
    focused_headroom,
    ENSEMBLE_DIR / "ceiling_focused_normalized_headroom_heatmap.png",
    "Ceiling-focused comparison: normalized headroom",
)

focused_summary = ensemble_headroom_summary.loc[ceiling_focus_strategies].reset_index()
focused_summary["label"] = focused_summary["strategy"].map(ceiling_focus_labels)
plt.figure(figsize=(9, 4.5))
sns.barplot(
    data=focused_summary,
    x="label",
    y="mean_normalized_headroom",
    color="#4C78A8",
)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Strategy")
plt.ylabel("Mean normalized headroom")
plt.title("How much baseline-to-ceiling headroom each ensemble captures")
plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / "ceiling_focused_mean_normalized_headroom.png", dpi=200)
plt.show()

ceiling_delta_vs_convex = (
    ensemble_normalized_headroom.loc["per_target_ceiling_aware_validation"]
    - ensemble_normalized_headroom.loc["per_target_convex_validation"]
).to_frame("delta_vs_per_target_convex")
ceiling_delta_vs_convex["metric"] = ceiling_delta_vs_convex.index
plt.figure(figsize=(9, 4))
sns.barplot(
    data=ceiling_delta_vs_convex,
    x="metric",
    y="delta_vs_per_target_convex",
    color="#F58518",
)
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=25, ha="right")
plt.xlabel("Test dataset / metric")
plt.ylabel("Ceiling-aware minus per-target convex")
plt.title("Where ceiling-aware weighting helps or hurts")
plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / "ceiling_aware_delta_vs_per_target_convex.png", dpi=200)
plt.show()

ceiling_weights = weights_df[weights_df["strategy"] == "per_target_ceiling_aware_validation"].copy()
if not ceiling_weights.empty:
    ceiling_weight_matrix = ceiling_weights.set_index("target_dataset")[DATASETS].reindex(index=DATASETS, columns=DATASETS)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        ceiling_weight_matrix,
        annot=True,
        fmt=".2f",
        cmap="mako",
        vmin=0,
        vmax=max(0.6, float(ceiling_weight_matrix.max().max())),
        linewidths=0.5,
        linecolor="white",
    )
    plt.xlabel("UdonPred training head")
    plt.ylabel("Target validation dataset")
    plt.title("Ceiling-aware ensemble weights")
    plt.tight_layout()
    plt.savefig(ENSEMBLE_DIR / "ceiling_aware_weight_heatmap.png", dpi=200)
    plt.show()
else:
    print("No ceiling-aware weights found; skipping ceiling-aware weight heatmap.")

ceiling_report_table = pd.DataFrame(
    {
        "raw_score": ensemble_matrix.loc["per_target_ceiling_aware_validation"],
        "normalized_headroom": ensemble_normalized_headroom.loc["per_target_ceiling_aware_validation"],
        "delta_vs_best_individual": ensemble_delta.loc["per_target_ceiling_aware_validation"],
        "delta_headroom_vs_per_target_convex": ceiling_delta_vs_convex["delta_vs_per_target_convex"],
    }
)
ceiling_report_table.to_csv(ENSEMBLE_DIR / "ceiling_aware_metric_report.csv")
display(ceiling_report_table.style.format("{:.3f}"))


## 9. Interpretation Draft


In [ ]:
non_oracle = ensemble_matrix.drop(index="best_individual_test_oracle")
best_strategy_by_metric = non_oracle.idxmax(axis=0)
best_score_by_metric = non_oracle.max(axis=0)

print("Q4 report notes:")
print("- Best individual heads by test metric are an optimistic oracle comparison because the head is selected using test performance.")
print("- Simple mean, validation-selected head, global convex, and global ridge are non-leaky when validation predictions are used for selection/fitting.")
print("- Per-target convex, ceiling-aware, and ridge strategies are non-leaky if their weights are fit from validation data; ceiling-aware weighting is exploratory when ceilings come from test-set overlaps.")
print()
print("Best non-oracle ensemble per metric:")
for metric in METRIC_COLS:
    print(f"- {metric}: {best_strategy_by_metric[metric]} ({best_score_by_metric[metric]:.3f}); "
          f"best individual {best_individual_scores[metric]:.3f} from {best_individual_heads[metric]}")

global_rows = ["global_convex_validation", "global_ridge_stacking_validation"]
print()
print("Single global ensemble generalization:")
for row in global_rows:
    stats = ensemble_summary.loc[row]
    print(
        f"- {row}: mean delta {stats['mean_delta_vs_best_individual']:.3f}, "
        f"wins {int(stats['wins_vs_best_individual'])}/{len(METRIC_COLS)}, "
        f"worst delta {stats['worst_delta']:.3f}."
    )

print()
print("Normalized headroom summary:")
display(ensemble_headroom_summary.style.format("{:.3f}"))
